<a href="https://colab.research.google.com/github/MohdAkeeb89622/DATA-SCIENCE-CAPSTONE-PROJECT/blob/main/Capstone_Project_(Stock_Market_Anomaly_Detection).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Mount Drive + set paths

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/Stock Market Anomaly Detection"

OUT_DIR = DATA_DIR + "/outputs"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#Imports + install

!pip -q install pandas numpy scikit-learn

import os
import numpy as np
import pandas as pd

In [3]:
#Verify folder structure

print("stocks exists:", os.path.exists(DATA_DIR + "/stocks"))
print("etfs exists:", os.path.exists(DATA_DIR + "/etfs"))

print("sample stocks:", os.listdir(DATA_DIR + "/stocks")[:5])
print("sample etfs:", os.listdir(DATA_DIR + "/etfs")[:5])

os.makedirs(OUT_DIR, exist_ok=True)
print("Outputs will be saved to:", OUT_DIR)

stocks exists: True
etfs exists: True
sample stocks: ['SLP.csv', 'SLB.csv', 'SIX.csv', 'SKY.csv', 'SMSI.csv']
sample etfs: ['JQUA.csv', 'KRE.csv', 'JPLS.csv', 'KMED.csv', 'KALL.csv']
Outputs will be saved to: /content/drive/MyDrive/Stock Market Anomaly Detection/outputs


In [4]:
#Data loader (reads CSVs for selected tickers)
"""read_ticker_csv:-
Reads one CSV for one ticker.
Ensures it has required columns.
Parses and cleans dates.
Renames columns to a standard format.
Adds a ticker column.
Returns a tidy DataFrame with consistent columns.

load_universe:-
For a list of tickers, looks for each ticker’s CSV under stocks/ or etfs/.
Uses read_ticker_csv to load and standardize each one.
Concatenates everything into a single, sorted DataFrame of all tickers."""


REQUIRED_COLS = ["Date","Open","High","Low","Close","Adj Close","Volume"]

def read_ticker_csv(path: str, ticker: str) ->pd.DataFrame:
  df = pd.read_csv(path)

  missing = [c for c in REQUIRED_COLS if c not in df.columns]
  if missing:
    raise ValueError(f"{ticker}: missing columns {missing} in {path}")

  df = df.copy()
  df["Date"] = pd.to_datetime(df["Date"], errors= "coerce")
  df = df.dropna(subset=["Date"])

  df = df.rename(columns={
      "Date": "date",
      "Open":"open",
      "High":"high",
      "Low":"low",
      "Close":"close",
      "Adj Close":"adj_close",
      "Volume":"volume"
  })

  df["ticker"] = ticker
  return df[["date","ticker","open","high","low","close","adj_close","volume"]]

def load_universe(data_dir: str, tickers: list) -> pd.DataFrame:
  frames = []
  for t in tickers:
    candidates = [
        os.path.join(data_dir, "stocks", f"{t}.csv"),
        os.path.join(data_dir, "etfs", f"{t}.csv"),
    ]

    path = next((p for p in candidates if os.path.exists(p)), None)
    if not path:
            raise FileNotFoundError(f"Missing {t}. Looked in: {candidates}")
    frames.append(read_ticker_csv(path, t))


  df = pd.concat(frames, ignore_index = True)
  return df.sort_values(["ticker", "date"]).reset_index(drop=True)



In [5]:
#choose your universe + load

UNIVERSE = ["QQQ", "AAPL", "MSFT", "NVDA", "AMZN", "META"]

df = load_universe(DATA_DIR, UNIVERSE)

print("Rows:", len(df), "Tickers:", df["ticker"].nunique())
df.head()

Rows: 36866 Tickers: 6


,date,ticker,open,high,low,close,adj_close,volume
0,1980-12-12,AAPL,0.513393,0.515625,0.513393,0.513393,0.406782,117258400.0
1,1980-12-15,AAPL,0.488839,0.488839,0.486607,0.486607,0.385558,43971200.0
2,1980-12-16,AAPL,0.453125,0.453125,0.450893,0.450893,0.357260,26432000.0
3,1980-12-17,AAPL,0.462054,0.464286,0.462054,0.462054,0.366103,21610400.0
4,1980-12-18,AAPL,0.475446,0.477679,0.475446,0.475446,0.376715,18362400.0


In [8]:
#Feature engineering (leakage-safe)
"""rolling_zscore:- “How many standard deviations is today’s value above/below the average of the last N days (excluding today)?”
range_percentile_past:- “What percentage of the last N days had a smaller range than today?”"""
"""This is the exact “no look-ahead” requirement"""

W_RETURN = 63
W_VOL = 21
W_RANGE = 63
MIN_HISTORY = max(W_RETURN, W_VOL, W_RANGE)

def rolling_zscore(series: pd.Series, window: int) -> pd.Series:
  """ Leakage-safe z-score:- mean/std computed using only past values via shift(1)"""
  mu = series.shift(1).rolling(window=window, min_periods=window).mean()
  sd = series.shift(1).rolling(window=window, min_periods=window).std(ddof=0)
  return (series - mu) / sd.replace(0, np.nan)

def range_percentile_past(range_series: pd.Series, window: int) -> pd.Series:
  x = range_series.to_numpy(dtype=float)
  out = np.full(len(x),np.nan, dtype=float)

  for i in range(len(x)):
    if i < window:
      continue

    past = x[i-window:i]
    if np.isnan(x[i]) or np.isnan(past).any():
      continue
    out[i] = (past < x[i]).mean()*100.0

  return pd.Series(out, index=range_series.index)




In [9]:
#Compute the 3 required features
"""what this block is doing:-
For each ticker, it builds a set of standardized, leakage-safe features:
1. ret – raw daily return
2. ret_z – return z-score vs last 63 days
3. volz – volume z-score vs last 21 days (log volume)
4. range_pct – intraday range percentile vs last 63 days
5. has_history – flag indicating when all rolling windows are “mature”
These are classic building blocks for a trading signal / alpha factor pipeline."""


df = df.copy()
df["ret"] = df.groupby("ticker")["adj_close"].pct_change()
df["ret_z"] = df.groupby("ticker")["ret"].apply(lambda s: rolling_zscore(s, W_RETURN)).reset_index(level=0, drop=True)

df["log_volume"] = np.log(df["volume"].replace(0, np.nan))
df["volz"] = df.groupby("ticker")["log_volume"].apply(lambda s: rolling_zscore(s, W_VOL)).reset_index(level=0, drop=True)

df["range"] = (df["high"] - df["low"]) / df["close"].replace(0, np.nan)
df["range_pct"] = df.groupby("ticker")["range"].apply(lambda s: range_percentile_past(s,W_RANGE)).reset_index(level=0, drop=True)

df["has_history"] = df.groupby("ticker").cumcount() >= MIN_HISTORY

df[df["has_history"]].head()



/tmp/ipython-input-4186554043.py:13: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["ret"] = df.groupby("ticker")["adj_close"].pct_change()


,date,ticker,open,high,low,close,adj_close,volume,ret,ret_z,log_volume,volz,range,range_pct,has_history
63,1981-03-16,AAPL,0.412946,0.417411,0.412946,0.412946,0.327194,9307200.0,0.039325,NaN,16.046299,0.948702,0.010811,98.412698,True
64,1981-03-17,AAPL,0.433036,0.437500,0.433036,0.433036,0.343111,10936800.0,0.048649,1.360598,16.207644,1.089751,0.010309,95.238095,True
65,1981-03-18,AAPL,0.459821,0.464286,0.459821,0.459821,0.364335,9234400.0,0.061856,1.667600,16.038446,0.766485,0.009709,90.476190,True
66,1981-03-19,AAPL,0.457589,0.457589,0.455357,0.455357,0.360798,9452800.0,-0.009709,-0.286216,16.061822,0.727854,0.004902,58.730159,True
67,1981-03-20,AAPL,0.459821,0.464286,0.459821,0.459821,0.364335,3651200.0,0.009804,0.250090,15.110566,-0.572514,0.009709,88.888889,True


In [10]:
#Rule-based detector (stock-day anomalies)
#Apply PDF thresholds + label type + why

RET_THR = 2.5
VOL_THR = 2.5
RANGE_THR = 95

scored = df[df["has_history"]].copy()

tr_ret = scored["ret_z"].abs() > RET_THR
tr_vol = scored["volz"] > VOL_THR
tr_rng = scored["range_pct"] > RANGE_THR

scored["anomaly_flag"] = (tr_ret | tr_vol | tr_rng).astype(int)

def label_type(row):
  types = []
  if pd.notna(row["ret_z"]) and abs(row["ret_z"]) > RET_THR:
    types.append("crash" if row["ret"] < 0 else "spike")
  if pd.notna(row["volz"]) and row["volz"] > VOL_THR:
    types.append("volume_shock")
  if not types and pd.notna(row["range_pct"]) and row["range_pct"] > RANGE_THR:
    types.append("range_spike")
  return " + ".join(types)

def label_why(row):
  reasons = []
  if pd.notna(row["ret_z"]) and abs(row["ret_z"]) > RET_THR:
    reasons.append("|ret_z| > 2.5")
  if pd.notna(row["volz"]) and row["volz"] > VOL_THR:
    reasons.append("volz > 2.5")
  if pd.notna(row["range_pct"]) and row["range_pct"] > RANGE_THR:
    reasons.append("range_pct > 95")
  return "; ".join(reasons)


scored["type"] = scored.apply(label_type, axis=1)
scored["why"] = scored.apply(label_why, axis=1)

scored["anomaly_flag"].value_counts()

"""we built a clean anomaly-detection pipeline:

Feature engineering per ticker:-
ret_z (return z-score),
volz (volume z-score),
range_pct (range percentile).

Thresholding:-flag “extreme” events in any dimension.

Binary anomaly label:-anomaly_flag ∈ {0,1}.

Semantic classification:-type tells you what kind of anomaly (crash/spike/volume_shock/range_spike/etc.).

Rule explanation:- why tells you which thresholds were broken."""

'we built a clean anomaly-detection pipeline:\n\nFeature engineering per ticker:-\nret_z (return z-score),\nvolz (volume z-score),\nrange_pct (range percentile).\n\nThresholding:-flag “extreme” events in any dimension.\n\nBinary anomaly label:-anomaly_flag ∈ {0,1}.\n\nSemantic classification:-type tells you what kind of anomaly (crash/spike/volume_shock/range_spike/etc.).\n\nRule explanation:- why tells you which thresholds were broken.'

In [11]:
#Market-day anomaly (market return + breadth)

tmp = scored.dropna(subset=["ret"]).copy()
g = tmp.groupby("date")

market = pd.DataFrame({
    "date": g["ret"].mean().index,
    "market_ret": g["ret"].mean().values,
    "breadth": g.apply(lambda x: (x["ret"] > 0).mean()).values
}).sort_values("date")

abs_m = market["market_ret"].abs()
thr = abs_m.shift(1).rolling(W_RETURN, min_periods = W_RETURN).quantile(0.95)

market["market_anomaly_flag"]= ((abs_m > thr) | (market["breadth"] < 0.30)).astype(int)

market["market_anomaly_flag"].value_counts()



"""*Collapses all per-stock returns into daily market metrics:-
equal-weighted market return (market_ret)
breadth (breadth = fraction of winners)

*Uses a rolling, leakage-safe 95th percentile to define what a “huge” market move is.

*Flags a day as a market anomaly when:-
the move is in the top ~5% of absolute moves over the last 63 trading days, or
breadth is very low (< 30% of stocks up), indicating a widespread selloff.

*This pairs nicely with your per-stock anomaly flags—you can later analyze:-
stock anomalies that happen on normal vs abnormal market days,
or filter out stock “anomalies” that are just part of a broad market move."""


/tmp/ipython-input-3210669784.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "breadth": g.apply(lambda x: (x["ret"] > 0).mean()).values


'*Collapses all per-stock returns into daily market metrics:-\nequal-weighted market return (market_ret)\nbreadth (breadth = fraction of winners)\n\n*Uses a rolling, leakage-safe 95th percentile to define what a “huge” market move is.\n\n*Flags a day as a market anomaly when:-\nthe move is in the top ~5% of absolute moves over the last 63 trading days, or\nbreadth is very low (< 30% of stocks up), indicating a widespread selloff.\n\n*This pairs nicely with your per-stock anomaly flags—you can later analyze:-\nstock anomalies that happen on normal vs abnormal market days,\nor filter out stock “anomalies” that are just part of a broad market move.'

In [12]:
#Daily Anomaly Card + Market-Day Table (save to Drive)

daily_anomaly_card = scored[[
    "date","ticker","anomaly_flag","ret","ret_z","volz","range_pct","type","why"
]].copy()

daily_anomaly_card["date"] = daily_anomaly_card["date"].dt.strftime("%Y-%m-%d")

market_out = market.copy()
market_out["date"] = market_out["date"].dt.strftime("%Y-%m-%d")

daily_path = os.path.join(OUT_DIR, "daily_anomaly_card.csv")
market_path = os.path.join(OUT_DIR, "market_anomaly_card.csv")

daily_anomaly_card.to_csv(daily_path, index=False)
market_out.to_csv(market_path, index=False)

print("Saved:", daily_path)
print("Saved:", market_path)

"""
we now have two CSV outputs:-

daily_anomaly_card.csv – per (date, ticker):-
Contains anomaly flag, type, and reasons, plus key metrics.
Great for drilling into which stock did what on a given day.

market_day_table.csv – per date:-
Contains market-level return, breadth, market anomaly flag, etc.
Great for understanding which days were “crazy” at the index level and aligning that with individual stock anomalies."""

Saved: /content/drive/MyDrive/Stock Market Anomaly Detection/outputs/daily_anomaly_card.csv
Saved: /content/drive/MyDrive/Stock Market Anomaly Detection/outputs/market_anomaly_card.csv


'\nwe now have two CSV outputs:-\n\ndaily_anomaly_card.csv – per (date, ticker):-\nContains anomaly flag, type, and reasons, plus key metrics.\nGreat for drilling into which stock did what on a given day.\n\nmarket_day_table.csv – per date:-\nContains market-level return, breadth, market anomaly flag, etc.\nGreat for understanding which days were “crazy” at the index level and aligning that with individual stock anomalies.'

In [17]:
#Date query function + example
"""This function is a little “query tool” to inspect one specific date:
it tells you what the overall market did that day and which tickers were anomalous."""

def date_query(date_str: str, daily_df: pd.DataFrame, market_df: pd.DataFrame):
  m = market_df[market_df["date"] == date_str]
  if m.empty:
    print("No market row for:", date_str)
    return

  m = m.iloc[0]
  print("=== Market Status ===")
  print("date:", date_str)
  print("market_ret:", float(m["market_ret"]))
  print("breadth:", float(m["breadth"]))
  print("market_anomaly_flag:", int(m["market_anomaly_flag"]))

  print("\n===Anonalous Tickers(rule) ===")
  rows = daily_df[(daily_df["date"] == date_str) & (daily_df["anomaly_flag"] == 1)]

  if rows.empty:
        print("None")
  else:
    display(rows[["ticker","type","ret","ret_z","volz","range_pct","why"]].sort_values("ticker"))

# Example (change date as you want)
date_query("2020-02-27", daily_anomaly_card, market_out)


"""
“What happened on this day?”

The date_query function:-
Looks up the market row for date_str:-
prints market_ret, breadth, market_anomaly_flag.

Filters daily_anomaly_card for that date and anomaly_flag == 1:-
shows all anomalous tickers with type, ret, ret_z, volz, range_pct, why.

Example:-
date_query("2020-02-27", daily_anomaly_card, market_out)

Tells you:-
Was it a market anomaly day?
Which stocks crashed/spiked/had volume shocks?
Exactly why each was flagged.
"""


=== Market Status ===
date: 2020-02-27
market_ret: -0.05458029783461756
breadth: 0.0
market_anomaly_flag: 1

===Anonalous Tickers(rule) ===


,ticker,type,ret,ret_z,volz,range_pct,why
9884,AAPL,crash + volume_shock,-0.065368,-4.162027,2.747646,96.825397,|ret_z| > 2.5; volz > 2.5; range_pct > 95
15642,AMZN,crash,-0.048136,-3.180481,1.416625,100.000000,|ret_z| > 2.5; range_pct > 95
17622,META,range_spike,-0.037779,-2.329509,0.615173,95.238095,range_pct > 95
26206,MSFT,crash + volume_shock,-0.070459,-5.358750,2.935901,98.412698,|ret_z| > 2.5; volz > 2.5; range_pct > 95
31540,NVDA,range_spike,-0.055666,-2.468313,1.254529,98.412698,range_pct > 95
36841,QQQ,crash + volume_shock,-0.050074,-5.021920,2.645341,100.000000,|ret_z| > 2.5; volz > 2.5; range_pct > 95


'\n“What happened on this day?”\n\nThe date_query function:-\nLooks up the market row for date_str:-\nprints market_ret, breadth, market_anomaly_flag.\n\nFilters daily_anomaly_card for that date and anomaly_flag == 1:-\nshows all anomalous tickers with type, ret, ret_z, volz, range_pct, why.\n\nExample:-\ndate_query("2020-02-27", daily_anomaly_card, market_out)\n\nTells you:-\nWas it a market anomaly day?\nWhich stocks crashed/spiked/had volume shocks?\nExactly why each was flagged.\n'

In [18]:
#Monthly report generator + save

def monthly_report(month_str: str, daily_df: pd.DataFrame, market_df: pd.DataFrame) -> pd.DataFrame:

    dc = daily_df.copy()
    mt = market_df.copy()

    dc["date_dt"] = pd.to_datetime(dc["date"])
    mt["date_dt"] = pd.to_datetime(mt["date"])

    m = dc[dc["date_dt"].dt.strftime("%Y-%m") == month_str].copy()
    m = m[m["anomaly_flag"] == 1].copy()

    m = m.merge(mt[["date_dt","market_anomaly_flag"]], on="date_dt", how="left")

    out = m[["date","ticker","type","ret_z","volz","market_anomaly_flag","why"]].copy()
    out = out.rename(columns={"market_anomaly_flag":"mkt_flag"})
    return out.sort_values(["date","ticker"])

rep = monthly_report("2020-02", daily_anomaly_card, market_out)
display(rep.head(30))

rep_path = os.path.join(OUT_DIR, "monthly_report_2020-02.csv")
rep.to_csv(rep_path, index=False)
print("Saved:", rep_path)

"""
The monthly_report function gives you a monthly summary of all
stock-level anomalies, enriched with a flag telling you whether
each happened on a normal market day or an anomalous one.

So for each month, you can:-
See which tickers had weird behavior.
Know what kind of anomaly it was (crash, spike, volume shock, etc.).
Know how extreme it was (ret_z, volz).
See whether the whole market was also stressed (mkt_flag = 1) or if this was more idiosyncratic to that stock.
"""


,date,ticker,type,ret_z,volz,mkt_flag,why
0,2020-02-03,AAPL,range_spike,-0.512873,1.188092,0,range_pct > 95
7,2020-02-03,AMZN,range_spike,-0.310841,1.233254,0,range_pct > 95
19,2020-02-03,MSFT,range_spike,2.463173,0.654027,0,range_pct > 95
20,2020-02-04,MSFT,spike,3.265823,1.382177,1,|ret_z| > 2.5; range_pct > 95
42,2020-02-04,QQQ,spike,3.032841,0.556518,1,|ret_z| > 2.5
21,2020-02-05,MSFT,range_spike,-0.500119,1.544442,0,range_pct > 95
8,2020-02-07,AMZN,range_spike,0.862085,0.754828,0,range_pct > 95
22,2020-02-10,MSFT,range_spike,2.211772,0.963288,0,range_pct > 95
30,2020-02-10,NVDA,volume_shock,2.437682,2.989664,0,volz > 2.5
23,2020-02-11,MSFT,crash,-2.593497,2.479131,0,|ret_z| > 2.5; range_pct > 95


Saved: /content/drive/MyDrive/Stock Market Anomaly Detection/outputs/monthly_report_2020-02.csv


'\nThe monthly_report function gives you a monthly summary of all \nstock-level anomalies, enriched with a flag telling you whether \neach happened on a normal market day or an anomalous one.\n\nSo for each month, you can:-\nSee which tickers had weird behavior.\nKnow what kind of anomaly it was (crash, spike, volume shock, etc.).\nKnow how extreme it was (ret_z, volz).\nSee whether the whole market was also stressed (mkt_flag = 1) or if this was more idiosyncratic to that stock.\n'

In [19]:
#Split evaluation (Train/Val/Test) + target anomaly rate check

def flag_rate(daily_df: pd.DataFrame, start: str, end: str) -> float:
    x = daily_df.copy()
    x["date_dt"] = pd.to_datetime(x["date"])
    x = x[(x["date_dt"] >= start) & (x["date_dt"] <= end)]
    return float((x["anomaly_flag"] == 1).mean())

print("Train 2018 flag rate:", flag_rate(daily_anomaly_card, "2018-01-01", "2018-12-31"))
print("Val   2019 flag rate:", flag_rate(daily_anomaly_card, "2019-01-01", "2019-12-31"))
print("Test 2020Q1 flag rate:", flag_rate(daily_anomaly_card, "2020-01-01", "2020-03-31"))

"""
This is a quick sanity check on your anomaly system:-
Is the anomaly rate stable across train/val?
Does it spike in known stressful periods (like 2020Q1)?
"""



Train 2018 flag rate: 0.12815405046480743
Val   2019 flag rate: 0.07671957671957672
Test 2020Q1 flag rate: 0.34139784946236557


'\nThis is a quick sanity check on your anomaly system:-\nIs the anomaly rate stable across train/val?\nDoes it spike in known stressful periods (like 2020Q1)?\n'